# VMD-MFGNN v2 — Full-Scale Colab Run (2010-01-01 to 2025-12-31)

This notebook packages the VMD-MFGNN copper-price-forecasting pipeline (Phase-1
code-fix-gate-verified) for a real, full-scale run on Google Colab with a GPU
runtime. It downloads real Yahoo Finance data for the 10 locked tickers, runs
the leakage-safe expanding-window VMD decomposition, trains the proposed
VMD-MFGNN model plus the 6 locked baselines (ARIMA, XGBoost, LSTM,
Transformer, VMD-LSTM, SimpleMTGNN) across the 4 locked horizons (1/5/10/22
trading days), runs the 4 locked ablations, computes Diebold-Mariano
significance tests, and generates all paper figures.

**Locked scope reference:** `vmd-mfgnn-protocol/SKILL.md` in the repo.

**Full runbook:** see the markdown cell "RUNBOOK" below, and the companion
file `COLAB_RUNBOOK.md` at the repo root (same content, for offline reading).

**Checkpointing philosophy:** Google Drive is mounted first and used as the
persistent backbone. Every expensive stage (VMD decomposition, each trained
model, each ablation variant) is checked against Drive before being
(re)computed, and synced to Drive immediately after it completes. If Colab
disconnects, re-running the notebook from the top will skip everything
already completed and resume from the first missing artifact.


## RUNBOOK

### Expected runtime (best-effort estimates — see assumptions)

Assumptions: Colab T4 (16GB) or better GPU, `hidden_dim=64`, `num_gnn_layers=2`,
`num_heads=4`, `temporal_layers=2`, dataset is ~2,500 train-day windows
(2010-2019), batch_size=32 (~80 batches/epoch), `epochs=200` with
`patience=20` early stopping. These are architecturally small models
(hidden_dim 64) over a modest number of samples, so per-epoch cost is low;
the dominant cost is epoch *count*, gated by early stopping, which is
data-dependent and cannot be predicted exactly.

| Stage | Estimate | Notes |
|---|---|---|
| Data download (10 tickers, yfinance) | 1-3 min | Network-bound, small CSVs |
| VMD decomposition (expanding window, refit every 21 days, ~10 vars x ~4000 days -> ~190 refits/var) | 15-45 min | CPU-bound (`vmdpy`), no GPU benefit; this is the single biggest *fixed* cost since it is not parallelized across variables in the current pipeline |
| Hyperparameter optimization (optional, `hpo.enabled: true`) | +~20-55 min | Only runs if opted in via `configs/default.yaml`; see "Optional HPO stage" note below |
| VMD-MFGNN training | 10-30 min | Small model; bounded mostly by how many epochs run before early stopping |
| LSTM / Transformer / SimpleMTGNN / VMD-LSTM (each) | 5-20 min each | Similar order of magnitude to VMD-MFGNN |
| XGBoost | 2-8 min | One `XGBRegressor` per horizon (4 total), CPU-bound, not GPU-bound |
| ARIMA | 5-20 min | Walk-forward: refits a small ARIMA(p,d,q) per test-window (~750+ windows), CPU-bound; no GPU benefit. Could be the slowest baseline if the test set is large |
| Ablations (4 variants, each ~ one more training run) | 30-90 min total | Same order as the individual training runs above |
| Significance tests + figure generation | 1-5 min | Cheap, CPU-bound |
| **Total, rough (HPO off)** | **~2-5 hours** | Wide range because it is dominated by early-stopping epoch counts and VMD's fixed CPU cost, neither of which is precisely predictable ahead of time. Budget for at least one Colab disconnect over this window. |
| **Total, rough (HPO on)** | **~2.5-6 hours** | Adds the HPO stage above on top of the HPO-off total. |

Do not treat these numbers as guarantees — they are architecture-based
estimates, not measurements (this notebook was built without GPU or network
access in the packaging environment). Watch the actual per-epoch timing
printed by the training loop (`trainer.py` logs every 10 epochs) once running
and re-budget from there.

#### Optional HPO stage — runtime estimate reasoning

`configs/default.yaml`'s `hpo:` block defaults to `enabled: false` (no extra
cost). If set to `true` with the shipped defaults (`n_trials=15`,
`trial_epochs=25`), `run_hpo()` trains up to `15 * 25 = 375` "trial epochs"
of VMD-MFGNN (Optuna's median pruner will cut some trials short, so this is a
ceiling, not a guarantee). The main VMD-MFGNN training run above budgets for
up to `epochs=200`, and its estimated 10-30 min already covers that full
budget in the worst case. Scaling proportionally: `375 / 200 ≈ 1.875`, so
the HPO stage adds roughly `1.875 * (10-30 min) ≈ 19-56 min`, rounded here
to **~20-55 min**. In practice this is likely an overestimate since (a)
pruning stops unpromising trials early and (b) some trial configs (e.g.
`hidden_dim=32`) are cheaper per epoch than the default `hidden_dim=64`.

### Checkpointing (trainer-native)

VMD-MFGNN's and each ablation variant's training is checkpointed via
`VMDMFGNNTrainer.fit(..., checkpoint_path=...)` (added to `src/trainer.py`
after this notebook was first packaged): `fit()` itself checks whether the
given `.pt` path already exists, loads and skips training if so, and
otherwise saves the best-so-far state dict there as validation loss
improves. This replaced the notebook's original hand-rolled "check if a
`.pt` file exists, manually `torch.load`/`torch.save`" logic in the
VMD-MFGNN and ablation cells — that logic is now redundant and has been
removed, leaving less notebook-side code to maintain. The notebook's own
responsibility is now only to (a) point `checkpoint_path` at a location
under `results/checkpoints/` and (b) call `sync_to_drive(('results',))`
afterward, since the trainer only writes locally and has no notion of Google
Drive. The 6 baselines still use their own manual "predictions already on
disk" resume check (below), since baseline `.fit()` methods have no
`checkpoint_path` parameter.

### What to do if Colab disconnects

- **During data/VMD stage:** Re-run the "Data" cell. It checks Google Drive
  first for a previously-saved `vmd_modes.npy` + `vmd_modes_meta.json`; if
  found (and the metadata matches — same date range/params/data hash) it
  loads from Drive instead of recomputing. If the raw price CSV was already
  downloaded to Drive too, that is also reused. Worst case (nothing on Drive
  yet), you lose only this stage's progress, not later stages.
- **During the optional HPO stage:** `run_hpo()` itself has no resume/
  checkpoint support (each of its trials is short relative to the main
  training runs, so this hasn't been needed) — if interrupted, re-run the
  HPO cell and it restarts the whole search from trial 0. If you'd rather
  not lose partial progress, disable HPO for the retry and use
  `configs/default.yaml`'s default `model:`/`training:` values instead.
- **During training stage:** Re-run the training cell. VMD-MFGNN is resumed
  via `VMDMFGNNTrainer.fit(checkpoint_path=...)` (see "Checkpointing"
  above): if `results/checkpoints/vmd_mfgnn.pt` already exists (e.g.
  restored from Drive), training is skipped and those weights are loaded
  directly. Each of the 6 baselines is checked individually against Drive
  via its own predictions-exist check: if a baseline's prediction arrays
  already exist on Drive, training for that baseline is skipped and its
  test metrics are recomputed from the saved predictions instead. Only
  models that had not yet finished when the disconnect happened will
  retrain. **Caveat:** `src/trainer.py`'s own `run_all_experiments()` (used
  as-is nowhere in this notebook's main training cell — see the note in
  that cell) only writes `results/all_results.json` once, at the very end,
  after every model finishes; this notebook does NOT call that function
  directly for the main training stage specifically so that per-model
  resume is possible. It is used as-is for ablations' final JSON dump
  structure only.
- **During ablation stage:** Same idea — each of the 4 ablation variants
  (`full_model`, `no_vmd_raw_price`, `pooled_graph`, `correlation_graph`) is
  checkpointed via `VMDMFGNNTrainer.fit(checkpoint_path=...)` pointed at
  `results/checkpoints/{name}.pt` (the same paths `run_ablation_studies()`
  itself uses), then `results/` is synced to Drive after each variant
  finishes. `run_ablation_studies()` from `src/trainer.py` itself has no
  built-in per-variant checkpointing (it only calls `save_results()` once at
  the end), so this notebook reimplements its 4 variants inline (importing
  the exact same model/trainer classes it uses) — it does not modify
  `src/trainer.py`.
- **During figures/significance stage:** Cheap to just re-run; nothing here
  is checkpointed individually since the whole stage should take well under
  5 minutes once the model and predictions already exist.

### Known risks to watch for

1. **Zinc/nickel ticker substitution.** `src/data_pipeline.py` uses
   `^NQCIZNER` (zinc) and `^NQCINIER` (nickel) — NASDAQ Commodity sub-indices
   — because no standalone Yahoo Finance futures ticker exists for LME-only
   zinc/nickel (`ZNC=F`/`NI=F` do not resolve to real instruments). **Verify,
   once you have real network access, that these two tickers actually return
   non-empty daily data for the full 2010-2025 range** (the "Data" cell below
   logs each ticker's returned row count — check the log for `no data
   returned` or `failed` warnings for these two specifically). If either
   comes back empty/short, the fallback plan (not implemented here, decide
   before re-running) is to either (a) drop that variable from the 10-var
   set and document a 9-variable scope deviation, or (b) find and substitute
   another genuine Yahoo Finance-listed proxy — do not silently reintroduce
   `ZNC=F`/`NI=F`.
2. **NaNs in the significance table.** At tiny smoke-test scale, prior
   verification runs saw 2/24 NaN entries in `results/significance_table.json`
   (likely `diebold_mariano_test`'s `var_d <= 0` degeneracy guard tripping on
   a very small/degenerate sample — see `src/utils.py`). This may or may not
   reproduce at full scale (full scale has far more test-set samples per
   horizon, which should make the HAC variance estimate much less degenerate)
   — **after the real run, open `results/significance_table.json` and check
   for any `NaN` / `null` `dm_stat`/`p_value` entries before treating the
   paper's significance claims as final.**
3. This notebook does not modify `src/*.py`, `scripts/*.py`, or
   `configs/*.yaml` — it imports and orchestrates them as-is. If you need to
   change hyperparameters, edit `configs/default.yaml` in the repo (or the
   uploaded/cloned copy) before running, not in this notebook. To opt into
   HPO, set `hpo.enabled: true` in that same file before running.
4. **HPO tunes VMD-MFGNN only** (see the STEP 1.5 markdown cell) — this is a
   deliberate, disclosed scope limitation, not an oversight.

## Setup 1/4 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/vmd_mfgnn_v2_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive results root:', DRIVE_ROOT)


Mounted at /content/drive
Drive results root: /content/drive/MyDrive/vmd_mfgnn_v2_results


## Setup 2/4 — Get the repo

Pick ONE of the two cells below (both are provided; uncomment the one you
want and leave the other commented). Either way you end up with the repo at
`/content/copper`.


In [2]:
# ---- OPTION A: git clone from a remote you set up (recommended) ----
# Fill in REPO_URL with your own remote (e.g. a private GitHub repo you pushed
# this codebase to), then uncomment the two lines below.

REPO_URL = 'https://github.com/anmol0705/Copper_Price_Forecasting'  # e.g. 'https://github.com/<you>/copper.git'

import subprocess
if not os.path.exists('/content/copper'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/copper'], check=True)
else:
    print('/content/copper already exists, skipping clone')


In [3]:
# ---- OPTION B: upload a copper.zip (use if you don't have a git remote) ----
# 1. Zip the repo locally: `cd D:\copper && zip -r copper.zip . -x '.git/*'`
# 2. Run this cell, click "Choose Files", and select copper.zip.
# 3. Uncomment the extraction lines.

# from google.colab import files
# uploaded = files.upload()  # select copper.zip
# import zipfile
# with zipfile.ZipFile('copper.zip', 'r') as zf:
#     zf.extractall('/content/copper')
# print('Extracted to /content/copper')


In [ ]:
import os
assert os.path.isdir('/content/copper'), (
    "Repo not found at /content/copper -- run Option A or Option B above first "
    "(uncomment the lines in whichever cell matches how you're getting the repo)."
)
os.chdir('/content/copper')
import sys
if '/content/copper' not in sys.path:
    sys.path.insert(0, '/content/copper')
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))


cwd: /content/copper
['.git', '.gitignore', 'COLAB_RUNBOOK.md', 'ORCHESTRATOR_REPORT.md', 'PAPER_RESULTS_PLAN.md', 'RESEARCH_BLUEPRINT.md', 'STATUS.md', 'configs', 'copper_fundamentals.md', 'gnn_literature_review.md', 'launch-claude.bat', 'literature_gap_analysis.md', 'literature_review_copper_price_forecasting.md', 'notebooks', 'paper', 'requirements.txt', 'scripts', 'src', 'vmd-mfgnn-protocol', 'vmd_research.md']


## Setup 3/4 — Install dependencies

In [6]:
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 319, in build
    wheel_file = _build_one(
                 ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 193, in _build_one
    wheel_path = _build_one_inside_env(
                 ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/whee

In [7]:
!pip install -q vmdpy optuna yfinance torch-geometric

## Setup 4/4 — Confirm GPU runtime

In [8]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. In Colab: Runtime > Change runtime type > '
          'Hardware accelerator > GPU (T4 or better recommended). Training will '
          'be much slower on CPU and the runtime estimates in the RUNBOOK cell '
          'above do not apply.')


torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Drive sync helpers

Small helpers used throughout the notebook to mirror local `data/` and
`results/` into the Drive checkpoint folder, and to restore from Drive at the
start of a resumed session. Kept in the notebook (not `src/`) since they are
Colab-session plumbing, not part of the research pipeline.


In [9]:
import shutil
from pathlib import Path

LOCAL_ROOT = Path('/content/copper')
DRIVE_ROOT_P = Path(DRIVE_ROOT)

def sync_to_drive(subpaths=('data', 'results')):
    """Mirror the given local subdirectories into Drive. Cheap/incremental:
    only copies files that don't exist yet or whose size differs, so this is
    safe to call frequently (e.g. after every model finishes)."""
    for sub in subpaths:
        src = LOCAL_ROOT / sub
        if not src.exists():
            continue
        dst = DRIVE_ROOT_P / sub
        for root, dirs, filenames in os.walk(src):
            rel = Path(root).relative_to(src)
            (dst / rel).mkdir(parents=True, exist_ok=True)
            for fn in filenames:
                s = Path(root) / fn
                d = dst / rel / fn
                if (not d.exists()) or d.stat().st_size != s.stat().st_size:
                    shutil.copy2(s, d)
    print(f'[sync_to_drive] mirrored {subpaths} -> {DRIVE_ROOT_P}')

def restore_from_drive(subpaths=('data', 'results')):
    """Copy the given subdirectories FROM Drive back into the local repo
    (used at the start of a resumed session, before any recomputation)."""
    for sub in subpaths:
        src = DRIVE_ROOT_P / sub
        if not src.exists():
            continue
        dst = LOCAL_ROOT / sub
        for root, dirs, filenames in os.walk(src):
            rel = Path(root).relative_to(src)
            (dst / rel).mkdir(parents=True, exist_ok=True)
            for fn in filenames:
                s = Path(root) / fn
                d = dst / rel / fn
                if (not d.exists()) or d.stat().st_size != s.stat().st_size:
                    shutil.copy2(s, d)
    print(f'[restore_from_drive] restored {subpaths} <- {DRIVE_ROOT_P}')

# Pull back anything from a previous (possibly interrupted) session before
# doing any work below.
restore_from_drive()


[restore_from_drive] restored ('data', 'results') <- /content/drive/MyDrive/vmd_mfgnn_v2_results


In [12]:
import src.data_pipeline as dp

# Drop the confirmed-dead zinc/nickel tickers
dp.TICKERS = {k: v for k, v in dp.TICKERS.items() if k not in ("zinc", "nickel")}
dp.VARIABLE_NAMES = list(dp.TICKERS.keys())

# Defensive patch: yfinance sometimes returns Close as a 1-col DataFrame, not a Series
import pandas as pd
_orig_download = dp.DataDownloader.download
def _patched_download(self):
    import yfinance as yf
    frames = {}
    for name, ticker in dp.TICKERS.items():
        try:
            data = yf.download(ticker, start=self.start, end=self.end, progress=False, auto_adjust=True)
            close = data["Close"]
            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            close = close.dropna()
            if len(close) > 0:
                frames[name] = close
                print(f"  {name} ({ticker}): {len(close)} rows")
            else:
                print(f"  WARNING: {name} ({ticker}): no data")
        except Exception as e:
            print(f"  WARNING: {name} ({ticker}): failed - {e}")
    if len(frames) < len(dp.TICKERS):
        missing = set(dp.TICKERS) - set(frames)
        raise ValueError(f"Missing data for: {missing} — fix before proceeding")
    df = pd.DataFrame(frames).ffill(limit=5).dropna()
    df.index.name = "date"
    self.cache_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(self.cache_path)
    print(f"Saved {len(df)} rows")
    return df
dp.DataDownloader.download = _patched_download

# delete any partial cache from the failed run before retrying
import os
if os.path.exists("data/raw_prices.csv"):
    os.remove("data/raw_prices.csv")

## STEP 1: Data Download & VMD Decomposition

Runs the real (non-debug) pipeline: full 2010-01-01..2025-12-31 range, real
Yahoo Finance downloads for all 10 locked tickers, and the leakage-safe
**expanding-window** VMD (`debug_fast=False` — never the batch/leaky
`VMDDecomposerFast` path). This mirrors `create_datasets(config)` exactly as
called in `scripts/run_experiments.py`.

`build_vmd_modes()` (in `src/data_pipeline.py`) already has its own on-disk
cache (`data/vmd_modes.npy` + `data/vmd_modes_meta.json`, keyed by a hash of
the input price data plus the VMD params/split dates) — `restore_from_drive()`
above already pulled any prior cache back from Drive, so if this stage
previously completed, `create_datasets` will detect the matching cache and
load it instantly instead of recomputing. Watch the log: `"Loading cached VMD
modes from ..."` means it skipped recomputation; `"VMD decomposing ..."`
per-variable log lines mean it's doing the real (slow) computation.


In [13]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)

from src.utils import load_config, set_seed
from src.data_pipeline import create_datasets

config = load_config('configs/default.yaml')
set_seed(config['training']['seed'])

print('Date range:', config['data']['start_date'], '->', config['data']['end_date'])
print('Tickers:', config['data']['tickers'])
print('VMD K:', config['vmd']['K'], '| refit_interval:', config['vmd']['refit_interval'])


Date range: 2010-01-01 -> 2025-12-31
Tickers: {'copper': 'HG=F', 'aluminum': 'ALI=F', 'zinc': '^NQCIZNER', 'nickel': '^NQCINIER', 'gold': 'GC=F', 'oil': 'CL=F', 'dxy': 'DX-Y.NYB', 'sp500': '^GSPC', 'vix': '^VIX', 'us10y': '^TNX'}
VMD K: 5 | refit_interval: 21


In [14]:
# Real run: debug_fast=False (leakage-safe expanding-window VMD). This is the
# expensive step (see RUNBOOK) -- it's what gets checkpointed to Drive
# immediately below.
data = create_datasets(config, debug_fast=False)

print('Variables:', data['variable_names'])
print('Train/Val/Test samples:', len(data['train_ds']), len(data['val_ds']), len(data['test_ds']))

# Sanity-check the zinc/nickel proxy tickers actually came back with data --
# see RUNBOOK risk (1). If either is missing from variable_names, the
# download silently dropped it (see DataDownloader.download in
# src/data_pipeline.py, which only keeps tickers that returned len(data) > 0).
for risky in ('zinc', 'nickel'):
    if risky in data['variable_names']:
        print(f'OK: {risky} present in downloaded data.')
    else:
        print(f'*** WARNING: {risky} is MISSING from data["variable_names"] -- '
              f'its Yahoo ticker likely returned no data. See RUNBOOK risk (1) '
              f'for the fallback plan. ***')


  copper (HG=F): 4023 rows
  aluminum (ALI=F): 2895 rows
  gold (GC=F): 4022 rows
  oil (CL=F): 4023 rows
  dxy (DX-Y.NYB): 4024 rows
  sp500 (^GSPC): 4023 rows
  vix (^VIX): 4023 rows
  us10y (^TNX): 4021 rows
Saved 2914 rows
Variables: ['copper', 'aluminum', 'gold', 'oil', 'dxy', 'sp500', 'vix', 'us10y']
Train/Val/Test samples: 1343 505 984
*** WARNING: zinc is MISSING from data["variable_names"] -- its Yahoo ticker likely returned no data. See RUNBOOK risk (1) for the fallback plan. ***
*** WARNING: nickel is MISSING from data["variable_names"] -- its Yahoo ticker likely returned no data. See RUNBOOK risk (1) for the fallback plan. ***


In [15]:
# Checkpoint immediately: VMD decomposition is the most expensive fixed cost
# in the whole pipeline (see RUNBOOK) and must survive a disconnect.
sync_to_drive(('data',))


[sync_to_drive] mirrored ('data',) -> /content/drive/MyDrive/vmd_mfgnn_v2_results


## STEP 1.5: Hyperparameter Optimization (optional, VMD-MFGNN only)

Controlled by the `hpo:` block in `configs/default.yaml` (`enabled`,
`n_trials`, `trial_epochs`). This is opt-in and defaults to `enabled: false`,
so by default this cell is a no-op and the pipeline behaves exactly as
before. `load_config('configs/default.yaml')` in the STEP 1 cell above
already loaded this key -- no notebook change was needed for that part.

**Scope limitation (deliberate, disclosed in the paper): HPO tunes VMD-MFGNN
only.** The 6 locked baselines (ARIMA/XGBoost/LSTM/Transformer/VMD-LSTM/
SimpleMTGNN) and all 4 ablation variants (`full_model`, `no_vmd_raw_price`,
`pooled_graph`, `correlation_graph`) are never touched by HPO -- they always
train with the plain `configs/default.yaml` `model:`/`training:` settings, in
both this notebook and the library's own `run_all_experiments()` /
`run_ablation_studies()`. This asymmetry exists because tuning every model
would multiply the already-long full-scale runtime by roughly (baselines +
ablations + 1) and the paper's novelty claim rests on VMD-MFGNN vs. baselines
at reasonable, not maximally-tuned-for-VMD-MFGNN-only, settings -- see
`src/hpo.py`'s module docstring for the same rationale.

If `config['hpo']['enabled']` is `True`, the cell below calls
`run_hpo(config, data, n_trials=config['hpo']['n_trials'],
trial_epochs=config['hpo']['trial_epochs'])` (an Optuna TPE search with
median pruning over `hidden_dim`, `num_heads`, `dropout`, `num_gnn_layers`,
`learning_rate`), prints the winning hyperparameters, and builds
`vmd_mfgnn_config` -- a copy of `config` with `model`/`training.learning_rate`
overridden by the HPO winner -- which the STEP 2 training cell below uses
when constructing the VMD-MFGNN model and trainer. This exactly mirrors the
override logic `run_all_experiments()` itself performs internally when
`config["hpo"]["enabled"]` is `True` (see `src/trainer.py`), so this
notebook's manually-driven training loop and the library's
`run_all_experiments()` path produce equivalent behavior whether HPO is on or
off. When disabled, `vmd_mfgnn_config` is simply `config` unchanged.

**Where results are saved for the human to inspect / report in the paper:**
- `results/hpo_best_params.json` -- the winning hyperparameter dict
- `results/hpo_trials.csv` -- full Optuna trial history (all trials, params,
  values, pruned/completed state)

Both are written by `run_hpo()` itself (see `src/hpo.py`) and are synced to
Drive by the cell below like everything else under `results/`.

In [16]:
import copy

if config.get('hpo', {}).get('enabled', False):
    from src.hpo import run_hpo

    n_trials = config['hpo'].get('n_trials', 15)
    trial_epochs = config['hpo'].get('trial_epochs', 25)
    print(f'HPO enabled: running Optuna search ({n_trials} trials, '
          f'{trial_epochs} epochs/trial) for VMD-MFGNN only...')

    best_params = run_hpo(config, data, n_trials=n_trials, trial_epochs=trial_epochs)
    print('HPO winner hyperparameters:', best_params)

    # Mirrors src/trainer.py's run_all_experiments() HPO-override block
    # exactly (same fields overridden, same source dict), so this notebook's
    # manually-driven training loop and the library's run_all_experiments()
    # path produce equivalent behavior when HPO is enabled.
    vmd_mfgnn_config = copy.deepcopy(config)
    vmd_mfgnn_config['model']['hidden_dim'] = best_params['hidden_dim']
    vmd_mfgnn_config['model']['num_heads'] = best_params['num_heads']
    vmd_mfgnn_config['model']['dropout'] = best_params['dropout']
    vmd_mfgnn_config['model']['num_gnn_layers'] = best_params['num_gnn_layers']
    vmd_mfgnn_config['training']['learning_rate'] = best_params['learning_rate']

    print('VMD-MFGNN will train with the HPO-tuned model/training config above.')
    print('Baselines and ablation variants below are unaffected (see markdown cell above).')
    sync_to_drive(('results',))  # persist hpo_best_params.json / hpo_trials.csv
else:
    print("HPO disabled (config['hpo']['enabled']=False in configs/default.yaml) -- "
          "VMD-MFGNN will train with the hidden_dim/num_heads/dropout/num_gnn_layers/"
          "learning_rate values from configs/default.yaml as-is.")
    vmd_mfgnn_config = config

HPO disabled (config['hpo']['enabled']=False in configs/default.yaml) -- VMD-MFGNN will train with the hidden_dim/num_heads/dropout/num_gnn_layers/learning_rate values from configs/default.yaml as-is.


## STEP 2: Train VMD-MFGNN + all 6 baselines (per-model checkpointed)

`src/trainer.py`'s `run_all_experiments(config, data)` trains VMD-MFGNN then
all 6 baselines sequentially and, importantly, **already saves each model's
prediction arrays to `results/predictions/{name}_{h}.npy` immediately after
that model finishes** (verified by reading `trainer.py`: `run_baseline()`
saves right after `model.fit()+model.predict()` for each baseline, and the
VMD-MFGNN block saves its predictions right after `trainer.fit()` too). What
it does NOT do is save the aggregated `results/all_results.json` until every
model has finished (`save_results(all_results, ...)` is the last line), and
it has no way to skip a model that was already trained in a previous,
disconnected session.

To get true per-model resume without editing `src/trainer.py`, this cell
**does not call `run_all_experiments()` directly**. Instead it drives the
exact same building blocks `run_all_experiments()` uses internally
(`VMDMFGNNTrainer`, `run_baseline`, the same `base_cfg` construction, the
same model classes) in a loop that, for each model, first checks whether
Drive already has that model's finished predictions and skips straight to
recomputing test metrics from the saved arrays if so. This is exactly the
"wrap the model loop with a resume check" strategy flagged as acceptable in
the packaging brief.

VMD-MFGNN specifically now gets its resume behavior for free from
`VMDMFGNNTrainer.fit(..., checkpoint_path=...)` (trainer-native, added after
this notebook was first packaged) instead of a hand-rolled
exists-check/`torch.load`/`torch.save` block: `fit()` itself checks whether
`results/checkpoints/vmd_mfgnn.pt` exists, loads and skips training if so,
and otherwise saves the best-so-far state dict there as it trains. The
6-baseline loop directly below still does its own manual "predictions
already on disk" check, since baseline `.fit()` methods have no
`checkpoint_path` parameter to delegate to.

At the end this produces the same `all_results` dict shape and the same
`results/all_results.json` / `results/predictions/*` / `results/interpretability/*`
artifacts that `run_all_experiments()` would have produced.

In [17]:
from pathlib import Path
import time
import numpy as np
import torch

from src.utils import compute_metrics, get_device, save_results, set_seed
from src.trainer import VMDMFGNNTrainer, run_baseline, run_significance_tests, print_results_table
from src.models.vmd_mfgnn import VMDMFGNN
from src.models.baselines import (
    LSTMBaseline, TransformerBaseline, VMDLSTMBaseline,
    SimpleMTGNN, XGBoostBaseline, ARIMABaseline,
)

set_seed(config['training']['seed'])
device = str(get_device())
horizons = data['horizons']
num_vars = data['num_vars']
num_modes = data['num_modes']

base_cfg = {
    'num_vars': num_vars, 'num_modes': num_modes,
    'hidden_dim': config['model']['hidden_dim'],
    'num_layers': config['model']['temporal_layers'],
    'num_heads': config['model']['num_heads'],
    'dropout': config['model']['dropout'],
    'lookback': config['data']['lookback'],
    'horizons': horizons,
    'epochs': config['training']['epochs'],
    'lr': config['training']['learning_rate'],
    'weight_decay': 1e-5,
    'patience': config['training']['patience'],
    'device': device,
}

pred_dir = Path('results/predictions')
interp_dir = Path('results/interpretability')
pred_dir.mkdir(parents=True, exist_ok=True)
interp_dir.mkdir(parents=True, exist_ok=True)

all_results = {}

def _predictions_exist(name, horizons):
    return all(
        (pred_dir / f'{name}_{h}.npy').exists() and (pred_dir / f'{name}_{h}_true.npy').exists()
        for h in horizons
    )

def _metrics_from_saved_predictions(name, horizons):
    res = {'name': name}
    for h in horizons:
        pred = np.load(pred_dir / f'{name}_{h}.npy')
        true = np.load(pred_dir / f'{name}_{h}_true.npy')
        res[f'h{h}'] = compute_metrics(true, pred)
    return res

print('base_cfg:', base_cfg)


base_cfg: {'num_vars': 8, 'num_modes': 5, 'hidden_dim': 64, 'num_layers': 2, 'num_heads': 4, 'dropout': 0.1, 'lookback': 60, 'horizons': [1, 5, 10, 22], 'epochs': 200, 'lr': 0.001, 'weight_decay': 1e-05, 'patience': 20, 'device': 'cuda'}


In [18]:
# ---- VMD-MFGNN (our model) ----
# Checkpointing is trainer-native (VMDMFGNNTrainer.fit(checkpoint_path=...)):
# if results/checkpoints/vmd_mfgnn.pt already exists (e.g. restored from Drive
# by restore_from_drive() above), fit() loads it and skips training entirely;
# otherwise it trains normally, saving the best-so-far state dict to that path
# every time validation loss improves (see src/trainer.py's fit() docstring).
# sync_to_drive() below is still needed afterward -- the trainer only writes
# locally to results/checkpoints/, it has no notion of Google Drive.
vmd_mc = vmd_mfgnn_config['model']
vmd_mfgnn_checkpoint = Path('results/checkpoints/vmd_mfgnn.pt')

model = VMDMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=vmd_mc['hidden_dim'], num_heads=vmd_mc['num_heads'],
    num_gnn_layers=vmd_mc['num_gnn_layers'],
    temporal_layers=vmd_mc['temporal_layers'],
    dropout=vmd_mc['dropout'], horizons=horizons,
    graph_type=vmd_mc['graph_type'],
)
trainer = VMDMFGNNTrainer(model, vmd_mfgnn_config)

print('Training VMD-MFGNN (proposed model)...')
t0 = time.time()
history = trainer.fit(data['train_loader'], data['val_loader'],
                       checkpoint_path=vmd_mfgnn_checkpoint)
if history.get('resumed_from_checkpoint'):
    print(f"[resume] loaded checkpoint from {vmd_mfgnn_checkpoint} -- training skipped")
else:
    print(f'VMD-MFGNN training done in {time.time() - t0:.1f}s')
sync_to_drive(('results',))

test_results = trainer.evaluate(data['test_loader'])
test_results['name'] = 'VMD-MFGNN'
all_results['VMD-MFGNN'] = test_results
print('VMD-MFGNN results:', test_results)

# predict() also (re)populates the model's internal attention/graph state
# needed for get_attention_weights()/get_learned_graphs() below, so always
# run it even when weights were loaded from a checkpoint.
vmd_preds = trainer.predict(data['test_loader'])
test_ys = []
for _, y in data['test_loader']:
    test_ys.append(y.numpy() if isinstance(y, torch.Tensor) else y)
test_true = np.concatenate(test_ys, axis=0)
for i, h in enumerate(horizons):
    np.save(pred_dir / f'VMD-MFGNN_{h}.npy', vmd_preds[str(h)])
    np.save(pred_dir / f'VMD-MFGNN_{h}_true.npy', test_true[:, i])

learned_graphs = model.get_learned_graphs()
attention_weights = model.get_attention_weights()
if learned_graphs:
    torch.save(learned_graphs, interp_dir / 'learned_graphs.pt')
if attention_weights is not None:
    torch.save(attention_weights, interp_dir / 'attention_weights.pt')

sync_to_drive(('results',))

Training VMD-MFGNN (proposed model)...
VMD-MFGNN training done in 4702.9s
[sync_to_drive] mirrored ('results',) -> /content/drive/MyDrive/vmd_mfgnn_v2_results
VMD-MFGNN results: {'h1': {'rmse': np.float64(0.01828548782952975), 'mae': 0.012461022473871708, 'mape': np.float32(152.72298), 'r2': -0.048560142517089844, 'da': np.float64(48.47560975609756)}, 'h5': {'rmse': np.float64(0.040247133752791715), 'mae': 0.028982141986489296, 'mape': np.float32(149.81999), 'r2': -0.018265962600708008, 'da': np.float64(49.59349593495935)}, 'h10': {'rmse': np.float64(0.05632388821856815), 'mae': 0.04202153906226158, 'mape': np.float32(194.84142), 'r2': -0.042726755142211914, 'da': np.float64(46.646341463414636)}, 'h22': {'rmse': np.float64(0.07852151205757275), 'mae': 0.061463743448257446, 'mape': np.float32(209.25958), 'r2': -0.07415270805358887, 'da': np.float64(48.06910569105691)}, 'avg_mse': np.float64(0.0028230497700860724), 'name': 'VMD-MFGNN'}
[sync_to_drive] mirrored ('results',) -> /content/dr

In [19]:
# ---- Raw-price + VMD baselines, each independently resumable ----
raw_baselines = [
    ('ARIMA', lambda: ARIMABaseline(base_cfg), data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader']),
    ('LSTM', lambda: LSTMBaseline(base_cfg), data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader']),
    ('Transformer', lambda: TransformerBaseline(base_cfg), data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader']),
    ('SimpleMTGNN', lambda: SimpleMTGNN(base_cfg), data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader']),
    ('XGBoost', lambda: XGBoostBaseline(base_cfg), data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader']),
    ('VMD-LSTM', lambda: VMDLSTMBaseline(base_cfg), data['train_loader'], data['val_loader'], data['test_loader']),
]

for name, make_model, train_loader, val_loader, test_loader in raw_baselines:
    if _predictions_exist(name, horizons):
        print(f'[resume] {name}: predictions already on disk -- skipping training, recomputing metrics')
        res = _metrics_from_saved_predictions(name, horizons)
    else:
        bl = make_model()
        res = run_baseline(bl, train_loader, val_loader, test_loader, name)
        sync_to_drive(('results',))
    all_results[name] = res
    print(name, '->', {k: v for k, v in res.items() if k != 'name'})


: 

In [ ]:
# ---- Save aggregated results + significance table (mirrors the tail end of
# run_all_experiments(), just called from here so it runs after the resumable
# loop above instead of only inside a single non-resumable function call) ----
save_results(all_results, 'results/all_results.json')
print('All results saved to results/all_results.json')

print_results_table(all_results, horizons)

baseline_names = [name for name, *_ in raw_baselines]
significance_table = run_significance_tests('VMD-MFGNN', baseline_names, horizons)

sync_to_drive(('results',))


In [ ]:
# Verify results/significance_table.json was actually produced (don't assume
# -- confirm) and flag any NaN entries per RUNBOOK risk (2).
import json as _json
sig_path = Path('results/significance_table.json')
assert sig_path.exists(), 'results/significance_table.json was not created!'
with open(sig_path) as f:
    sig_table = _json.load(f)
print(f'significance_table.json has {len(sig_table)} entries')

nan_entries = [
    k for k, v in sig_table.items()
    if v.get('dm_stat') is None or v.get('p_value') is None
    or (isinstance(v.get('dm_stat'), float) and v['dm_stat'] != v['dm_stat'])  # NaN check
    or (isinstance(v.get('p_value'), float) and v['p_value'] != v['p_value'])
]
if nan_entries:
    print(f'*** WARNING: {len(nan_entries)}/{len(sig_table)} entries have NaN dm_stat/p_value: {nan_entries} ***')
    print('See RUNBOOK risk (2) -- sanity-check whether this is expected at full scale.')
else:
    print('No NaN entries in significance_table.json.')


## STEP 3: Ablation studies (4 variants, each checkpointed)

`src/trainer.py`'s `run_ablation_studies(config, data)` trains the 4 locked
ablation variants (`full_model`, `no_vmd_raw_price`, `pooled_graph`,
`correlation_graph` -- see `vmd-mfgnn-protocol/SKILL.md`) sequentially and
only calls `save_results(ablation_results, 'results/ablation_results.json')`
once, at the very end -- it has no per-variant checkpointing at all, unlike
`run_baseline()` used in STEP 2. So the same resume strategy is applied here:
this cell reimplements the 4 variants inline (importing the exact same
classes `run_ablation_studies()` uses: `VMDMFGNN`, `PooledGraphMFGNN`,
`_UnsqueezeModeWrapper`, `VMDMFGNNTrainer`). Per-variant resumability now
comes for free from `VMDMFGNNTrainer.fit(..., checkpoint_path=...)`
(trainer-native, same mechanism used for VMD-MFGNN in STEP 2): each variant's
`fit()` call is pointed at `results/checkpoints/{name}.pt` -- the exact same
paths `run_ablation_studies()` itself uses -- so a variant whose checkpoint
is already present (e.g. restored from Drive) skips training entirely and
just re-evaluates, while a new variant trains normally and checkpoints as it
goes. `results/` is synced to Drive after each variant finishes.
`src/trainer.py` itself is not modified.

In [ ]:
from src.models.pooled_graph_mfgnn import PooledGraphMFGNN
from src.trainer import _UnsqueezeModeWrapper

set_seed(config['training']['seed'])
mc = config['model']  # raw config, NOT vmd_mfgnn_config -- ablations are
                       # never HPO-tuned (see STEP 1.5 markdown cell)
ablation_results = {}
checkpoint_dir = Path('results/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# (a) full_model: standard VMD-MFGNN, learned per-band graphs, real VMD data.
# Checkpointing is trainer-native: fit(checkpoint_path=...) loads and skips
# training if results/checkpoints/full_model.pt already exists (e.g. restored
# from Drive), otherwise trains and saves the best-so-far weights there.
name = 'full_model'
full_model = VMDMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=mc['hidden_dim'], num_heads=mc['num_heads'],
    num_gnn_layers=mc['num_gnn_layers'], temporal_layers=mc['temporal_layers'],
    dropout=mc['dropout'], horizons=horizons, graph_type='learned',
)
full_trainer = VMDMFGNNTrainer(full_model, config)
full_trainer.fit(data['train_loader'], data['val_loader'],
                  checkpoint_path=checkpoint_dir / f'{name}.pt')
full_results = full_trainer.evaluate(data['test_loader'])
full_results['name'] = name
ablation_results[name] = full_results
sync_to_drive(('results',))
print(name, '->', full_results)

In [ ]:
# (b) no_vmd_raw_price: genuine no-VMD baseline on raw-price loaders (never
# VMD-decomposed), via a num_modes=1 VMDMFGNN wrapped to unsqueeze a singleton
# mode dim -- matches src/trainer.py's _UnsqueezeModeWrapper usage exactly.
# Checkpointing is trainer-native (see (a) above).
name = 'no_vmd_raw_price'
no_vmd_inner = VMDMFGNN(
    num_vars=num_vars, num_modes=1,
    hidden_dim=mc['hidden_dim'], num_heads=mc['num_heads'],
    num_gnn_layers=mc['num_gnn_layers'], temporal_layers=mc['temporal_layers'],
    dropout=mc['dropout'], horizons=horizons, graph_type='learned',
)
no_vmd_model = _UnsqueezeModeWrapper(no_vmd_inner)
no_vmd_trainer = VMDMFGNNTrainer(no_vmd_model, config)
no_vmd_trainer.fit(data['raw_train_loader'], data['raw_val_loader'],
                    checkpoint_path=checkpoint_dir / f'{name}.pt')
no_vmd_results = no_vmd_trainer.evaluate(data['raw_test_loader'])
no_vmd_results['name'] = name
ablation_results[name] = no_vmd_results
sync_to_drive(('results',))
print(name, '->', no_vmd_results)

In [ ]:
# (c) pooled_graph: PooledGraphMFGNN -- ONE pooled graph instead of K
# per-band graphs. Highest-priority ablation per the locked protocol.
# Checkpointing is trainer-native (see (a) above).
name = 'pooled_graph'
pooled_model = PooledGraphMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=mc['hidden_dim'], num_heads=mc['num_heads'],
    num_gnn_layers=mc['num_gnn_layers'], temporal_layers=mc['temporal_layers'],
    dropout=mc['dropout'], horizons=horizons, graph_type='learned',
)
pooled_trainer = VMDMFGNNTrainer(pooled_model, config)
pooled_trainer.fit(data['train_loader'], data['val_loader'],
                    checkpoint_path=checkpoint_dir / f'{name}.pt')
pooled_results = pooled_trainer.evaluate(data['test_loader'])
pooled_results['name'] = name
ablation_results[name] = pooled_results
sync_to_drive(('results',))
print(name, '->', pooled_results)

In [ ]:
# (d) correlation_graph: VMDMFGNN with graph_type='correlation', per-band
# precomputed correlation adjacency threaded through via the trainer's
# _forward/_compute_precomputed_adjs helper (unchanged, used as-is).
# Checkpointing is trainer-native (see (a) above).
name = 'correlation_graph'
corr_model = VMDMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=mc['hidden_dim'], num_heads=mc['num_heads'],
    num_gnn_layers=mc['num_gnn_layers'], temporal_layers=mc['temporal_layers'],
    dropout=mc['dropout'], horizons=horizons, graph_type='correlation',
)
corr_trainer = VMDMFGNNTrainer(corr_model, config)
corr_trainer.fit(data['train_loader'], data['val_loader'],
                  checkpoint_path=checkpoint_dir / f'{name}.pt')
corr_results = corr_trainer.evaluate(data['test_loader'])
corr_results['name'] = name
ablation_results[name] = corr_results
sync_to_drive(('results',))
print(name, '->', corr_results)

# Final aggregated file matching what run_ablation_studies() itself would
# have produced at results/ablation_results.json.
save_results(ablation_results, 'results/ablation_results.json')
sync_to_drive(('results',))
print('Ablation results saved to results/ablation_results.json')

## STEP 4: Generate figures

Mirrors `scripts/run_experiments.py`'s `main()` exactly: pulls the trained
VMD-MFGNN model's attention weights (converting to numpy if it's still a
torch tensor) and calls `generate_all_figures(config, data, results,
ablation_results, output_dir=..., model=..., attn_weights=...)`.


In [ ]:
from src.visualize import generate_all_figures

attn_weights = trainer.model.get_attention_weights()
if hasattr(attn_weights, 'cpu'):
    attn_weights = attn_weights.cpu().numpy()
elif not isinstance(attn_weights, np.ndarray):
    attn_weights = np.array(attn_weights)

fig_dir = Path('results/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

generate_all_figures(
    config, data, all_results, ablation_results,
    output_dir=str(fig_dir),
    model=trainer.model,
    attn_weights=attn_weights,
)

print('Figures written to', fig_dir)
print(sorted(os.listdir(fig_dir)))

sync_to_drive(('results',))


## Final sync + download

Full final mirror of `data/` and `results/` to Drive (idempotent — safe to
re-run), plus an optional zip of `results/` for a one-click local download.


In [ ]:
sync_to_drive(('data', 'results'))
print('Final sync complete. Drive folder:', DRIVE_ROOT)
print(sorted(os.listdir(DRIVE_ROOT)))


In [ ]:
# Optional: zip results/ and download directly (in addition to the Drive copy).
import shutil as _shutil
_shutil.make_archive('/content/vmd_mfgnn_v2_results', 'zip', 'results')

from google.colab import files
files.download('/content/vmd_mfgnn_v2_results.zip')


---
### Done

Check, in order:
1. `results/all_results.json` — all 7 models x 4 horizons.
2. `results/significance_table.json` — DM test vs. each baseline; re-check for NaNs (RUNBOOK risk 2).
3. `results/ablation_results.json` — all 4 ablation variants.
4. `results/figures/` — all paper figures.
5. `results/predictions/`, `results/interpretability/` — raw arrays / learned graphs / attention weights for further analysis.

All of the above are mirrored to `/content/drive/MyDrive/vmd_mfgnn_v2_results/`
and will persist even if this Colab runtime is later recycled.
